# RSA: Ataque de Texto Conhecido

> **DLT — T. Nóbrega | UFCG**

Este notebook demonstra como o **RSA sem padding** é vulnerável ao **ataque de texto conhecido** (Known Plaintext Attack).

Por ser **determinístico**, a mesma mensagem sempre produz o mesmo criptograma. Isso permite que um atacante monte um dicionário e identifique mensagens sem precisar da chave privada.




## 1. Configuração das chaves 

números pequenos para didática

In [15]:
# Parâmetros RSA (exemplo do material de aula)
p = 61
q = 53
n = p * q          # módulo = 3233
e = 17             # expoente público
d = 2753           # expoente privado

print(f"=== Par de Chaves RSA ===")
print(f"Chave Pública ($K_pub$) : (n={n}, e={e})")
print(f"Chave Privada ($K_priv$) : (n={n}, d={d})")
print(f"p={p}, q={q}")

=== Par de Chaves RSA ===
Chave Pública ($K_pub$) : (n=3233, e=17)
Chave Privada ($K_priv$) : (n=3233, d=2753)
p=61, q=53


## 2. Funções de criptografia RSA

In [ ]:
def rsa_encrypt(m, e, n):
    return pow(m, e, n)

def rsa_decrypt(c, d, n):
    return pow(c, d, n)




Mensagem original  : 123
Criptograma        : 855
Mensagem decifrada : 123
Correto? True


In [9]:
m = 1234
c = rsa_encrypt(m, e, n)
print(f"Mensagem original  : {m}")
print(f"Criptograma        : {c}")

Mensagem original  : 1234
Criptograma        : 2183


In [10]:
m_dec = rsa_decrypt(c, d, n)
print(f"Mensagem decifrada : {m_dec}")
print(f"Correto? {m == m_dec}")

Mensagem decifrada : 1234
Correto? True


## 3. O Ataque

**montando o dicionário** : Alice sempre envia uma dentre 3 respostas possíveis: `"sim"`, `"nao"` ou `"talvez"`.

Eva (a atacante) conhece as possíveis mensagens e pode **pré-computar os criptogramas**.

In [11]:
mensagens_possiveis = {
    "sim":    ord('s') + ord('i') + ord('m'), 
    "nao":    ord('n') + ord('a') + ord('o'),
    "talvez": ord('t') + ord('a') + ord('l'),
}

vamos precomputar um dicionaro com a chave publica, $K_{pub} = n,e$

In [17]:
dicionario_ataque = {}
for texto, valor in mensagens_possiveis.items():
    criptograma = rsa_encrypt(valor, e, n)  
    dicionario_ataque[criptograma] = texto
    print(f"'{texto}' (m={valor}) ---> criptograma={criptograma}")

'sim' (m=329) ---> criptograma=358
'nao' (m=318) ---> criptograma=901
'talvez' (m=321) ---> criptograma=1476


Alice envia uma mensagem cifrada

In [18]:
mensagem_alice = "sim"
m_alice = mensagens_possiveis[mensagem_alice]
c_interceptado = rsa_encrypt(m_alice, e, n)
print(f"Alice enviou o criptograma: {c_interceptado}")

Alice enviou o criptograma: 358


 Eva consulta o dicionário

In [19]:
if c_interceptado in dicionario_ataque:
    print(f"Descobrimos: a mensagem é '{dicionario_ataque[c_interceptado]}'")
else:
    print("Mensagem não encontrada no dicionário.")

Descobrimos: a mensagem é 'sim'


## 4. Demonstração do determinismo

Cifrando a mesma mensagem **múltiplas vezes** — o resultado é sempre igual. Isso é o coração da vulnerabilidade.

In [21]:
m_teste = 42
resultados = [rsa_encrypt(m_teste, e, n) for _ in range(5)]
print(f"Cifrando m={m_teste} cinco vezes:")
for i, r in enumerate(resultados, 1):
    print(f"  Tentativa {i}: {r}")
print(f"\nTodos iguais? {len(set(resultados)) == 1} — isso torna o RSA puro vulnerável!")

Cifrando m=42 cinco vezes:
  Tentativa 1: 2557
  Tentativa 2: 2557
  Tentativa 3: 2557
  Tentativa 4: 2557
  Tentativa 5: 2557

Todos iguais? True — isso torna o RSA puro vulnerável!


## 5. A Contramedida: OAEP Padding

Com padding **OAEP** (Optimal Asymmetric Encryption Padding), a mesma mensagem gera criptogramas **diferentes a cada vez**, quebrando o determinismo.

In [22]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.backends import default_backend

In [27]:
# trecho gerado por ia para mostrar o RSA com OAEP
# Gerar par de chaves RSA real (2048 bits)
chave_privada = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
    backend=default_backend()
)
chave_publica = chave_privada.public_key()

mensagem = b"sim"
print(f"e={chave_publica.public_numbers().e}, n={chave_publica.public_numbers().n}")

e=65537, n=24218830483333227239429931782429454894199010121268727128051209467621825411260824341332234742725424725662494286897457579637693187096790555206299271102559533224680370390052556336414608469401039156369763620324387868142740295468918506087215675753887057566640588849803771976729588428208535903733030821223544113820522061266534751001775307906658026116404490161471423258457285138299064077572216227089104133788652859233837698839299290821220786688975588677595260291574491962539705876119758079206255569150977952351214496032830749197616486862118459974731582194565104810396843971565319004164473534408204184282373986299545616363801


In [32]:

print("=== RSA com OAEP é NÃO-DETERMINÍSTICO (seguro!) ===")
criptogramas = []
for i in range(5):
    c = chave_publica.encrypt(
        mensagem,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    criptogramas.append(c.hex()[:40])  # mostra só os primeiros bytes
    print(f"  Tentativa {i+1}: {c.hex()[:40]}...")

print(f"\nTodos iguais? {len(set(criptogramas)) == 1}")

=== RSA com OAEP é NÃO-DETERMINÍSTICO (seguro!) ===
  Tentativa 1: 674036dce9b3f617caf90248877f92b1d310ed37...
  Tentativa 2: 98f5ca634998b9386545bdabdc7f19a3730c4f9d...
  Tentativa 3: afa9273d4ac479b45d7910ae305642275db29579...
  Tentativa 4: 9140e3835443dc0c4f33c21304b1ba0aca3f65f9...
  Tentativa 5: 5f2df8ac20540841aa768947c76f1c085f3140a3...

Todos iguais? False


>  Nunca use RSA puro para criptografia de dados. Sempre utilize padding adequado (OAEP para criptografia, PSS para assinaturas).

---
*DLT — T. Nóbrega | UFCG*